In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, classification_report
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [2]:
DATA_PATH = "../../data/final_merged_dataset.parquet"

merged_dataset = pd.read_parquet(DATA_PATH)
merged_dataset = merged_dataset.sort_values("datetime_hour").reset_index(drop=True)

In [3]:
TARGET = "alarm_active"

In [4]:
excl_from_features = [
    "datetime_hour", "alarm_active", "alarm_minutes_in_hour",
    "region_key", "day_datetime", "date", "hour_datetime",
    "day_sunrise", "day_sunset", "region_id", "alarm_lag_1"
]

all_features = [c for c in merged_dataset.columns if c not in excl_from_features]

X = merged_dataset[all_features].select_dtypes(include=["number", "bool"]).copy()
y = merged_dataset[TARGET]

In [5]:
n = len(merged_dataset)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_valid = X.iloc[train_end:valid_end]
y_valid = y.iloc[train_end:valid_end]

X_test = X.iloc[valid_end:]
y_test = y.iloc[valid_end:]

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test :", X_test.shape)

Train: (597475, 459)
Valid: (128030, 459)
Test : (128031, 459)


In [6]:
nan_counts = X_train.isna().sum()
print(nan_counts[nan_counts > 0].sort_values(ascending=False))

kw_air_threat         5184
kw_sea_threat         5184
kw_energy_focus       5184
kw_shahed_activity    5184
dtype: int64


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
valid_proba = model.predict(X_valid_scaled)

best_threshold = 0.5
best_f1 = -np.inf

for thr in np.arange(0.20, 0.71, 0.05):
    preds = (valid_proba >= thr).astype(int)
    score = f1_score(y_valid, preds, zero_division=0)
    print(f"thr={thr:.2f} | F1={score:.4f}")
    if score > best_f1:
        best_f1 = score
        best_threshold = float(thr)

print(f"\nBest threshold: {best_threshold}")
print(f"Best valid F1:  {best_f1:.4f}")

In [ ]:
y_pred_raw = model.predict(X_test_scaled)
y_pred = (y_pred_raw >= best_threshold).astype(int)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_raw):.4f}")
print(classification_report(y_test, y_pred, target_names=["No alarm", "Alarm"]))

In [ ]:
BLUE_MAIN = "#2F6DB3"
RED_ACCENT = "#D1495B"

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_pred_raw[y_test == 0], bins=80, alpha=0.6, color=BLUE_MAIN, label="No alarm (actual)")
ax.hist(y_pred_raw[y_test == 1], bins=80, alpha=0.6, color=RED_ACCENT, label="Alarm (actual)")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1.5, label="Threshold = 0.35")
ax.set_xlabel("Predicted value")
ax.set_ylabel("Count")
ax.set_title("Linear Regression predictions distribution by actual class")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
print(y_pred_raw.min()) 
print(y_pred_raw.max())  

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns.tolist(),
    "coefficient": np.abs(model.coef_)
}).sort_values("coefficient", ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top15 = feature_importance.head(15)
ax.barh(top15["feature"], top15["coefficient"], color=BLUE_MAIN)
ax.set_xlabel("Absolute Coefficient")
ax.set_title("Top 15 Important Features")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
feature_importance_signed = pd.DataFrame({
    "feature": X.columns.tolist(),
    "coefficient": model.coef_
}).sort_values("coefficient", ascending=False)

positive = feature_importance_signed[feature_importance_signed["coefficient"] > 0].head(15)
negative = feature_importance_signed[feature_importance_signed["coefficient"] < 0].tail(15).sort_values("coefficient")

In [ ]:
plt.figure(figsize=(10, 7))
plt.barh(positive["feature"], positive["coefficient"], color=BLUE_MAIN)
plt.xlabel("Coefficient")
plt.title("Top positive coefficients")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
plt.barh(negative["feature"], negative["coefficient"].abs(), color=RED_ACCENT)
plt.xlabel("Absolute coefficient")
plt.title("Top negative coefficients")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()